Part A:
- generate training data
- save training data

---

IMPORTS:

In [7]:
import os
dir = os.path.abspath('')  # directory of notebook
import numpy as np
import pandas as pd

SYSTEM-LEVEL PARAMETERS:

In [8]:
Tamb = 21           # ambient temperature (C)
Tmax = 75           # maximumum temperature, i.e., steady state of u=100; set s.t. u=50 yields ~55 C
t0 = 0              # start time (s)
dt = 1              # time step (s)
t_ramp = 200  # 1000       # estimated time to reach steady state (s)
t_rest = 20  # 50         # time to "rest" at steady state for sake of obtaining dT=0 training data
t_drop = 400  # 800        # estimated time to return to Tamb (s)
n = 20              # number of step tests to obtain data for

DATA GENERATION:

In [9]:
u1_tests = np.linspace(100 / n, 100, n)  # heater power (%), const. values at which to obtain data
T_tests = Tamb + (Tmax - np.flip(np.logspace(np.log10(Tamb), np.log10(Tmax), n + 1))[1::])  # steady state temperature corresponding to u1_tests; decreases according to logscale as u increases

t_seg = t_ramp + t_rest + t_drop + t_rest / n  # length of single test
tf = t0 + n * t_seg
tvec = np.linspace(t0, tf, int((tf + 1) / dt))
dt = tvec[1] - tvec[0]  # redefine in case dt got rounded

u = np.zeros_like(tvec)
T = np.zeros_like(tvec) + Tamb

def T_approx(t_start, t_end, i):
    """Estimate temperature change using natural log of quadratic."""
    x = tvec[t_start:t_end] - tvec[t_start]
    H = T_tests[i] - Tamb
    W = t_end - t_start
    
    W0 = W / np.sqrt(1 - 1 / H)                 # == W'
    Q = H * (1 - ((x - W + W0) / W0 - 1) ** 2)  # == Q'
    L = np.log(Q) * H / np.log(H)               # == L'
    
    return L

t1 = int(t0)
for i in range(n):
    t2 = int(t1 + t_ramp)
    t3 = int(t1 + t_ramp + t_rest)
    t4 = int(t1 + t_ramp + t_rest + t_drop)
    
    T[t1:t2] = Tamb + T_approx(t1, t2, i)        # ramp up
    T[t2:t3] = T_tests[i]                        # steady state
    T[t3:t4] = T_tests[i] - T_approx(t3, t4, i)  # drop

    u[t1:t3] = u1_tests[i]  # const.

    t1 = int(t1 + t_seg)

T += np.random.rand(len(T))  # add noise

def u_cutoff(u):
    """Enforce [0, 100] bounds."""
    u[u < 0] = 0
    u[u > 100] = 100
    return u

u = u_cutoff(u)

SAVE:

In [11]:
data = pd.DataFrame(data={'t': tvec, 'T': T, 'u': u})
data.to_csv(os.path.join(dir, "data", "v11_training_data.csv"))